# build
Unpack ExeBench `.jsonl.zst` shards -> flat `stage1.jsonl` (cleaned of the `{"text": ...}` wrapper).

In [ ]:
import json, os
from pathlib import Path
import zstandard as zstd
from dotenv import load_dotenv

load_dotenv(Path("../../.env"))
SPLIT = os.environ["SPLIT"]
LIMIT = int(os.environ.get("LIMIT", "3000"))
ROOT  = Path("../../data")
SHARD_DIR = ROOT / "input" / SPLIT          # directory of .jsonl.zst shards
OUT       = ROOT / "build" / SPLIT / "stage1.jsonl"

def iter_records(shard_dir: Path):
    """Unwrapped ExeBench records from every .jsonl.zst (glob skips current_chunk_incomplete)."""
    shards = sorted(shard_dir.glob("*.jsonl.zst"))   # one level, ignores the partial file
    if not shards:
        raise SystemExit(f"no .jsonl.zst under {shard_dir}")
    print(f"{len(shards)} shards in {shard_dir}")
    for shard in shards:
        with zstd.open(open(shard, "rb"), "rt", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    yield json.loads(line)["text"]   # unwrap the {"text": ...} layer

In [ ]:
# === build stage1.jsonl: unpack + clean the shards into flat JSONL ===
OUT.parent.mkdir(parents=True, exist_ok=True)
kept = 0
with OUT.open("w") as out:
    for row in iter_records(SHARD_DIR):
        if not row.get("func_def"):
            continue
        rec = {
            "fname":     row.get("fname"),
            "func_def":  row["func_def"],
            "real_deps": row.get("real_deps"),
            "io_pairs":  row.get("real_io_pairs"),
            "ref":       row.get("ref"),     # provenance / license trail
            "path":      row.get("path"),
        }
        out.write(json.dumps(rec) + "\n")
        kept += 1
        if LIMIT and kept >= LIMIT:
            break
print(f"wrote {kept} records -> {OUT}")


In [ ]:
# === profile the built corpus ===
import re
from collections import Counter
rows = [json.loads(l) for l in OUT.open()]
test_pat = re.compile(r'(^|/)(test|tests|testfiles|testcases)(/|$)|cppcheck|autocorres', re.I)

n = len(rows)
print(f"total: {n}")
print(f"has a loop:         {sum(1 for r in rows if re.search(r'\b(for|while)\b', r['func_def']))}")
print(f"one-liner bodies:   {sum(1 for r in rows if r['func_def'].count(';') <= 1)}")
print(f"test/fixture paths: {sum(1 for r in rows if test_pat.search(r['path'] or ''))}")
print(f"empty real_deps:    {sum(1 for r in rows if not (r['real_deps'] or '').strip(' \n#1'))}")
repos = Counter("/".join((r['path'] or '').split('/')[:2]) for r in rows)
print(f"distinct repos: {len(repos)}  | top: {repos.most_common(5)}")
